In [ ]:
# ============================================================
# CELL 0 — Install dependencies (safe to re-run)
# ============================================================
# Run this once. subprocess keeps it out of notebook globals.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "google-genai", "openpyxl", "pandas"], check=True)
print("deps ok")

In [ ]:
# ============================================================
# CELL 1 — Paths and config
# ============================================================
import os, json, time, re
import pandas as pd
from openpyxl import load_workbook
from datetime import datetime

BASE     = "BASE"
OUT_DIR  = os.path.join(BASE, "LLM_Scores_Gemini")
os.makedirs(OUT_DIR, exist_ok=True)

# Bump this any time scoring logic/order changes — old JSONs get wiped automatically
SCHEMA_VERSION = 2

# Input files
FILES = {
    "Gemini_EvalForm_Part_1": os.path.join(BASE, "Gemini_EvalForm_Part_1 (Responses).xlsx"),
    "Gemini_EvalForm_Part_2": os.path.join(BASE, "Gemini_EvalForm_Part_2 (Responses).xlsx"),
}

# Which LLM annotator slot to fill — we are acting as L3 (Gemini)
# L1=col41, L2=col46, L3=col51, L4=col56  (0-indexed in raw sheet)
ANNOTATOR_SLOT = "L3"
SLOT_COL = {"L1": 41, "L2": 46, "L3": 51, "L4": 56}
SCORE_COL_START = SLOT_COL[ANNOTATOR_SLOT]  # C1 lands here, then C2,C3,C4,Total

# Data rows start at Excel row 4 (0-indexed row 3) — rows 0,1,2 are headers
DATA_ROW_START = 3   # 0-indexed pandas row index after reading with header=None

GEMINI_API_KEY = "API_KEY"

print("config ok")

In [ ]:
# ============================================================
# CELL 2 — The evaluation prompt (minimally adapted for LLM)
# ============================================================

SYSTEM_PROMPT = """You are an expert Odia (ଓଡ଼ିଆ) linguist and grammarian with deep specialization in spelling, script, grammatical, and error evaluation tasks. You are serving as an evaluator in an Odia GEC (Grammatical Error Correction) annotation study.

--- BACKGROUND (for your reference only — this work is already done) ---
Each Odia sentence in this study has been annotated for errors under exactly one of these five categories, applied in priority order:
1. Script Normalization — Unicode-level encoding errors: nukta misplacement, incorrect virama, vowel sign decomposition, ZWNJ/ZWJ issues, unintended ligatures.
2. Spelling & Typographical Errors — Correct encoding but wrong characters: short/long vowel confusion, phonetically similar consonant substitution, missing/extra chars, wrong word boundaries.
3. Grammatical Errors — Morphosyntactic issues: wrong verb tense/inflection, agreement errors, wrong case markers, word order, faulty copular constructions, missing punctuation.
4. Code-Mixing / Wrong Language — Roman-script words, non-Odia numerals, other Indic script characters, unnecessary loanwords.
5. Correct Sentence / No Errors — No errors present.

Annotation rules already applied:
- Each sentence contains at most one error; only the primary erroneous span is marked.
- Span selection covers only the minimal necessary erroneous unit.
- If no error exists: error span is empty, description states no error, corrected sentence equals the source.
- Priority rule: when a span could fit multiple categories, only the highest-priority one is assigned.
--- END BACKGROUND ---

--- YOUR TASK — EVALUATION ONLY ---
You will receive a source sentence alongside a submitted annotation response. The annotation contains five fields: Has Errors, Error Span, Annotated Category, Description, Corrected Sentence.

Your job is solely to judge how correct that submitted annotation is, using the four criteria below. Do not re-annotate the sentence. Do not second-guess what the correct answer should be independently — evaluate only what was submitted against the established annotation rules above.

C1 — ERROR EXISTS [0 or 1]
  1 : The submitted Has Errors flag correctly reflects whether an error is present in the source sentence.
  0 : Incorrect.

C2 — SPAN + DESCRIPTION [0, 1, or 2]
  2 : Exact span AND fully correct explanation (identifies what the error is, why it is wrong, and what the correct form should be).
  1 : Partial span and/or incomplete explanation.
  0 : Wrong span or hallucinated/irrelevant explanation. If no error: span must be empty and description must state no error exists.

C3 — ERROR CATEGORY [0 or 1]
  1 : The submitted category is correct (specific or parent-level acceptable). If no error exists, it must be "Correct Sentence / No Errors".
  0 : Incorrect.

C4 — CORRECTED SENTENCE [0, 1, or 2]
  2 : Fully correct, fluent, no new errors introduced.
  1 : Mostly correct with minor issues.
  0 : Incorrect, introduces new errors, or changes meaning. If no error: must match the source sentence exactly.

Also write ONE reason (10–30 words) summarising your overall judgment across all four components together.

Respond ONLY in this exact JSON format — no extra text, no markdown:
{
  "C1": <0 or 1>,
  "C2": <0, 1, or 2>,
  "C3": <0 or 1>,
  "C4": <0, 1, or 2>,
  "reason": "<10–30 word summary>"
}"""


def build_user_prompt(row):
    """Format one data row into the user message."""
    return f"""Source Sentence: {row['Source Sentence']}
Has Errors: {row['Has Errors']}
Error Span: {row['Error Span']}
Annotated Category: {row['Annotated Category']}
Description: {row['Description']}
Corrected Sentence: {row['Corrected Sentence']}"""

print("prompt ok")

In [ ]:
# ============================================================
# CELL 3 — Gemini client + call function
# ============================================================
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_ID = "gemini-3.1-flash-lite-preview"

def call_gemini(sentence_prompt, model_id=MODEL_ID):
    resp = client.models.generate_content(
        model=model_id,
        contents=sentence_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.0,
            top_p=1.0,
            max_output_tokens=4096,
        )
    )
    return resp.text.strip()


def parse_scores(raw_text):
    """Pull JSON out of the model response."""
    # strip markdown fences if present
    raw_text = re.sub(r"```(?:json)?", "", raw_text).strip().rstrip("`").strip()
    data = json.loads(raw_text)
    c1 = int(data["C1"])
    c2 = int(data["C2"])
    c3 = int(data["C3"])
    c4 = int(data["C4"])
    reason = str(data.get("reason", "")).strip()
    assert c1 in (0,1), f"C1 out of range: {c1}"
    assert c2 in (0,1,2), f"C2 out of range: {c2}"
    assert c3 in (0,1), f"C3 out of range: {c3}"
    assert c4 in (0,1,2), f"C4 out of range: {c4}"
    return c1, c2, c3, c4, reason

print("gemini client ok")

In [ ]:
# ============================================================
# CELL 4 — Resume helper: load / save progress JSON
# ============================================================

def json_path(file_key):
    return os.path.join(OUT_DIR, f"progress_{file_key}_gemini.json")

def load_progress(file_key):
    p = json_path(file_key)
    if os.path.exists(p):
        with open(p) as f:
            data = json.load(f)
        # If schema version doesn't match, old scores are stale — start fresh
        if data.get("__version__") != SCHEMA_VERSION:
            print(f"  [progress] stale schema in {p} — wiping and starting fresh")
            os.remove(p)
            return {}
        return data
    return {}

def save_progress(file_key, progress):
    progress["__version__"] = SCHEMA_VERSION
    with open(json_path(file_key), "w") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

print("resume helpers ok")

In [ ]:
# ============================================================
# ============================================================
# CELL 5 — Write output in Google Form response sheet style
# ============================================================
# The output mimics exactly what a human annotator produces
# when their Google Form responses are exported as a spreadsheet.
# One row = one model submission. Score values are label strings,
# not raw numbers — matching the human CSV format exactly.

C1_LABEL = {1: "1 — Correctly identified",        0: "0 — Incorrect identification"}
C2_LABEL = {2: "2 — Exact span AND fully correct explanation",
            1: "1 — Partial span and/or incomplete explanation",
            0: "0 — Wrong span or hallucinated/irrelevant explanation"}
C3_LABEL = {1: "1 — Correct category",             0: "0 — Incorrect category"}
C4_LABEL = {2: "2 — Fully correct, fluent, no new errors",
            1: "1 — Mostly correct with minor issues",
            0: "0 — Incorrect, introduces errors, or changes meaning"}

PROFILE_HEADERS = [
    "Timestamp", "Full Name", "Age", "Email ID",
    "Contact Information", "Odia Proficiency",
    "I voluntarily agree to participate in this evaluation study.",
]
# Repeated 50 times — one group per task page, exactly as in human CSV
TASK_HEADERS = (
    ["C1 — Is the error detection correct?",
     "C2 — Is the error span and description correct?",
     "C3 — Is the error category correct?",
     "C4 — Is the corrected sentence correct?"]
    * 50
)
ALL_HEADERS = PROFILE_HEADERS + TASK_HEADERS


def write_gform_response(src_path, out_path, progress, model_tag):
    """One flat row per model, matching Google Form export format."""
    from openpyxl import Workbook
    import pandas as pd

    # Read source to get Sentence_IDs for the reasons sheet
    df_raw = pd.read_excel(src_path, header=None)
    df = df_raw.iloc[DATA_ROW_START:].copy()
    df.columns = df_raw.iloc[2]
    df = df.reset_index(drop=True)

    wb = Workbook()
    ws = wb.active
    ws.title = "Form Responses"

    ws.append(ALL_HEADERS)

    # Profile fields — model metadata in human annotator slots
    profile_values = [
        datetime.now().strftime("%m/%d/%Y %H:%M:%S"),
        model_tag,       # "Full Name" slot
        "N/A",           # Age
        "N/A",           # Email
        "N/A",           # Contact
        "N/A",           # Odia Proficiency
        "Yes, I agree",  # Consent
    ]

    # 50 tasks x 4 scores = 200 score columns
    score_values = []
    for i in range(50):
        key = str(i)
        if key in progress:
            s = progress[key]
            score_values += [
                C1_LABEL[s["C1"]],
                C2_LABEL[s["C2"]],
                C3_LABEL[s["C3"]],
                C4_LABEL[s["C4"]],
            ]
        else:
            score_values += ["", "", "", ""]   # unevaluated rows left blank

    ws.append(profile_values + score_values)

    # Reasons sheet — one row per evaluated sentence
    ws_r = wb.create_sheet("Reasons")
    ws_r.append(["Row #", "Sentence_ID", "Source Sentence", f"Reason ({model_tag})"])
    for i in range(50):
        key = str(i)
        if key not in progress:
            continue
        sent_id  = df.iloc[i].get("Sentence_ID", f"row{i+1}")
        src_sent = df.iloc[i].get("Source Sentence", "")
        ws_r.append([i + 1, sent_id, src_sent, progress[key].get("reason", "")])

    wb.save(out_path)
    print(f"  saved -> {out_path}")

In [ ]:
# ============================================================
# ============================================================
# CELL 5 — Write output in Google Form response sheet style
# ============================================================
# The output mimics exactly what a human annotator produces
# when their Google Form responses are exported as a spreadsheet.
# One row = one model submission. Score values are label strings,
# not raw numbers — matching the human CSV format exactly.

C1_LABEL = {1: "1 — Correctly identified",        0: "0 — Incorrect identification"}
C2_LABEL = {2: "2 — Exact span AND fully correct explanation",
            1: "1 — Partial span and/or incomplete explanation",
            0: "0 — Wrong span or hallucinated/irrelevant explanation"}
C3_LABEL = {1: "1 — Correct category",             0: "0 — Incorrect category"}
C4_LABEL = {2: "2 — Fully correct, fluent, no new errors",
            1: "1 — Mostly correct with minor issues",
            0: "0 — Incorrect, introduces errors, or changes meaning"}

PROFILE_HEADERS = [
    "Timestamp", "Full Name", "Age", "Email ID",
    "Contact Information", "Odia Proficiency",
    "I voluntarily agree to participate in this evaluation study.",
]
# Repeated 50 times — one group per task page, exactly as in human CSV
TASK_HEADERS = (
    ["C1 — Is the error detection correct?",
     "C2 — Is the error span and description correct?",
     "C3 — Is the error category correct?",
     "C4 — Is the corrected sentence correct?"]
    * 50
)
ALL_HEADERS = PROFILE_HEADERS + TASK_HEADERS


def write_gform_response(src_path, out_path, progress, model_tag):
    """One flat row per model, matching Google Form export format."""
    from openpyxl import Workbook
    import pandas as pd

    # Read source to get Sentence_IDs for the reasons sheet
    df_raw = pd.read_excel(src_path, header=None)
    df = df_raw.iloc[DATA_ROW_START:].copy()
    df.columns = df_raw.iloc[2]
    df = df.reset_index(drop=True)

    wb = Workbook()
    ws = wb.active
    ws.title = "Form Responses"

    ws.append(ALL_HEADERS)

    # Profile fields — model metadata in human annotator slots
    profile_values = [
        datetime.now().strftime("%m/%d/%Y %H:%M:%S"),
        model_tag,       # "Full Name" slot
        "N/A",           # Age
        "N/A",           # Email
        "N/A",           # Contact
        "N/A",           # Odia Proficiency
        "Yes, I agree",  # Consent
    ]

    # 50 tasks x 4 scores = 200 score columns
    score_values = []
    for i in range(50):
        key = str(i)
        if key in progress:
            s = progress[key]
            score_values += [
                C1_LABEL[s["C1"]],
                C2_LABEL[s["C2"]],
                C3_LABEL[s["C3"]],
                C4_LABEL[s["C4"]],
            ]
        else:
            score_values += ["", "", "", ""]   # unevaluated rows left blank

    ws.append(profile_values + score_values)

    # Reasons sheet — one row per evaluated sentence
    ws_r = wb.create_sheet("Reasons")
    ws_r.append(["Row #", "Sentence_ID", "Source Sentence", f"Reason ({model_tag})"])
    for i in range(50):
        key = str(i)
        if key not in progress:
            continue
        sent_id  = df.iloc[i].get("Sentence_ID", f"row{i+1}")
        src_sent = df.iloc[i].get("Source Sentence", "")
        ws_r.append([i + 1, sent_id, src_sent, progress[key].get("reason", "")])

    wb.save(out_path)
    print(f"  saved -> {out_path}")

In [ ]:
# ============================================================
# CELL 6 — Run this cell to evaluate
# TEST_MODE=True  → first 10 rows only (safe to test with)
# TEST_MODE=False → all 50 rows (full run)
# Re-running always skips already-scored rows automatically.
# ============================================================
# ← flip to False when ready for full run

TEST_MODE  = True
TEST_LIMIT = 60

def evaluate_file(file_key, src_path, test_mode=True):
    print(f"\n{'='*60}")
    print(f"FILE: {file_key}  |  slot: {ANNOTATOR_SLOT}  |  test={test_mode}")
    print(f"{'='*60}")

    # Read the source file — skip the two header rows (rows 0 and 1 in 0-index)
    df_raw = pd.read_excel(src_path, header=None)

    # Row 2 (0-indexed) holds the real column names
    df = df_raw.iloc[DATA_ROW_START:].copy()
    df.columns = df_raw.iloc[2]          # use row-2 as column names
    df = df.reset_index(drop=True)

    # Keep only what the LLM is allowed to see
    visible_cols = ["Source Sentence", "Has Errors", "Error Span",
                    "Annotated Category", "Description", "Corrected Sentence"]
    df_visible = df[visible_cols].copy()

    limit = TEST_LIMIT if test_mode else len(df_visible)
    progress = load_progress(file_key)

    out_path = os.path.join(OUT_DIR, f"{file_key}_Gemini_GForm_Response.xlsx")

    for i in range(limit):
        key = str(i)
        if key in progress:
            print(f"  row {i+1:02d} — already done, skip")
            continue

        row = df_visible.iloc[i]
        sent_id = df.iloc[i].get("Sentence_ID", f"row{i+1}")
        print(f"  row {i+1:02d} | {sent_id} | calling Gemini ...", end=" ", flush=True)

        try:
            prompt = build_user_prompt(row)
            raw    = call_gemini(prompt)
            c1, c2, c3, c4, reason = parse_scores(raw)
            progress[key] = {"C1": c1, "C2": c2, "C3": c3, "C4": c4, "reason": reason}
            save_progress(file_key, progress)
            print(f"C1={c1} C2={c2} C3={c3} C4={c4} | {reason[:60]}")
        except Exception as e:
            print(f"FAILED — {e}")
            # leave it out of progress so it retries next run

        time.sleep(0.5)   # gentle rate limiting

    # Write gform-style response file
    write_gform_response(src_path, out_path, progress, model_tag=f"Gemini_{file_key}")
    print(f"\nDone. {len(progress)} rows scored.")


# Run
for fkey, fpath in FILES.items():
    evaluate_file(fkey, fpath, test_mode=TEST_MODE)